# Распознавание рукописных цифр MNIST с использованием CNN
Система распознавания рукописных цифр базы MNIST на основе **свёрточной нейронной сети**.
Требование: вероятность правильного распознавания на тестовой базе **> 99 %** — проверяется автоматически в ячейке 2.

Вывод для первых 10 цифр тестовой базы: столбец 1 — изображение, столбец 2 — что распознала нейросеть (и её уверенность), столбец 3 — что изображено на самом деле.

In [ ]:
# === Ячейка 1: загрузка данных и обучение ===
import numpy as np                 # работа с массивами (argmax)
import matplotlib.pyplot as plt    # рисование картинок и таблицы
import tensorflow as tf            # TensorFlow (в Colab уже установлен)

# Загружаем MNIST: 60 000 обучающих и 10 000 тестовых картинок 28x28
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Нормализация: пиксели 0..255 -> 0..1 (сети так учатся лучше)
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255

# Свёрточному слою нужна ось каналов: (N, 28, 28) -> (N, 28, 28, 1), 1 канал = оттенки серого
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

# Свёрточная нейросеть (архитектура из официального примера Keras "Simple MNIST convnet")
model = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28, 1)),                                  # картинка 28x28, 1 канал
    tf.keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),  # 32 фильтра 3x3 ищут простые черты (штрихи, края)
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),                     # уменьшаем карты признаков вдвое, оставляя самое яркое
    tf.keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),  # 64 фильтра ищут сочетания черт (дуги, петли, углы)
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),                     # ещё раз уменьшаем
    tf.keras.layers.Flatten(),                                          # вытягиваем карты признаков в вектор
    tf.keras.layers.Dropout(0.5),                                       # при обучении случайно отключаем половину — против переобучения
    tf.keras.layers.Dense(10, activation="softmax"),                    # 10 выходов = вероятности цифр 0..9
])

# Настраиваем обучение: оптимизатор Adam, функция потерь для меток-чисел, метрика точность
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# 15 эпох порциями по 128 картинок; 10% обучающих данных откладываем для контроля (валидации)
history = model.fit(x_train, y_train, batch_size=128, epochs=15, validation_split=0.1)

In [ ]:
# === Ячейка 2: проверка требований ===
model.summary()   # в списке слоёв видны Conv2D — значит сеть свёрточная

is_cnn = any(isinstance(layer, tf.keras.layers.Conv2D) for layer in model.layers)
loss, acc = model.evaluate(x_test, y_test, verbose=0)   # точность на всех 10 000 тестовых картинках

print(f"Сеть свёрточная: {is_cnn}")
print(f"Точность на тесте: {acc:.2%}")
assert is_cnn, "В сети нет свёрточных слоёв"
assert acc > 0.99, "Точность не превышает 99%"
print("Оба требования выполнены")

In [ ]:
# === Ячейка 3: предсказания для первых 10 тестовых цифр ===
probs = model.predict(x_test[:10])   # форма (10, 10): 10 вероятностей для каждой картинки
pred = np.argmax(probs, axis=1)      # что распознала сеть = номер максимальной вероятности
conf = np.max(probs, axis=1)         # уверенность сети = сама эта максимальная вероятность
true = y_test[:10]                   # что изображено на самом деле (правильные метки)
print("Сеть:       ", pred)
print("Правда:     ", true)
print("Уверенность:", np.round(conf * 100, 2), "%")

In [ ]:
# === Ячейка 4: таблица 10 строк x 3 столбца ===
fig, axes = plt.subplots(10, 3, figsize=(6, 16))   # сетка 10x3, размер в дюймах

# Заголовки столбцов (над первой строкой)
for col, name in enumerate(["Изображение", "Распознала сеть", "На самом деле"]):
    axes[0, col].set_title(name, fontsize=12)

for i in range(10):
    ok = pred[i] == true[i]                   # угадала ли сеть
    color = "green" if ok else "red"          # ошибку выделяем красным

    axes[i, 0].imshow(x_test[i, :, :, 0], cmap="gray")         # столбец 1: картинка (без оси канала)
    axes[i, 1].text(0.5, 0.5, f"{pred[i]} ({conf[i]:.2%})",    # столбец 2: ответ сети и уверенность
                    fontsize=18, ha="center", va="center", color=color)
    axes[i, 2].text(0.5, 0.5, str(true[i]),                    # столбец 3: правильный ответ
                    fontsize=28, ha="center", va="center")

    for ax in axes[i]:
        ax.axis("off")                        # убираем рамки и оси

plt.tight_layout()   # аккуратно раскладываем ячейки без наложений
plt.show()